<a href="https://colab.research.google.com/github/KinzaAsif2456/discoverey/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [73]:
%pip install -q duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":      f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":      f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily_march": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:20} {n:>12,} rows")

dim_clients                   104 rows
dim_content               519,606 rows
fact_daily_march        9,841,378 rows


## 1. Data Contract Summary

* **Grain:** (date, client_hash_id, content_hash_id)
* **Tables:** `fact_content_daily_performance`, `dim_clients`
* **Time Window:** March 2026 split into H1 (March 1–15, feature generation) and H2 (March 16–31, outcome window).
* **Target / Proxy:** `is_declining` = 1 if H2 impressions < 0.80 * H1 impressions, else 0.
* **Deliberately Excluded:** `fact_content_query_90d` (rolling 90-day window leaks future dates into the evaluation split).

### Field Mapping

| Category | Fields | Rationale |
|---|---|---|
| **Context** | `client_hash_id`, `content_hash_id`, `report_date` | Primary keys and join targets. |
| **Features (Mar 1–15)** | `imp_h1`, `avg_pos_h1`, `active_days_h1`, `ga4_coverage_h1`, `ctr_h1` | Strictly computed from H1 window before decision boundary. |
| **Label** | `imp_h2` | Computed from Mar 16–31 to derive target binary flag. |
| **Excluded** | `fact_content_query_90d` | Boundary overlap risk. |

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**What one row means for my lane:**
I’m looking at search performance, so for me, one row represents a single page (URL) for one of our clients on a specific day. I'll eventually be grouping these days together to see how a page changes over time, but the "grain" of the warehouse data is (date, client, content_item).

**Which table(s) I’ll use:**
My main source is fact_content_daily_performance. I'm focusing on the March 2026 data specifically for this foundation work. I’ll also pull in dim_clients just to see when they started tracking so I don't get confused by "new" accounts.

**Which time window:**
I’m splitting March 2026 into two blocks:
March 1–15: This is my "history" where I build my features.
March 16–31: This is the "future" where I see if the page actually declined.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**What I'd predict or rank (label / proxy).**
I’m creating a proxy called is_declining. If a page’s impressions in the second half of March drop by more than 20% compared to the first half, I’ll mark it as 1 (declining). It’s a simple way to flag pages that are losing visibility.

**One thing I deliberately exclude**
I’m ignoring the fact_content_query_90d table for now. Because it’s a "rolling 90-day" window, it might contain data from April or May that would "leak" the future into my March model. It’s better to stay safe and just use the daily facts.

**Field buckets:**

| Bucket | Fields | Why |
|---|---|---|
| Context (join/group only) | `client_hash_id`, `content_hash_id`, `report_date` | pseudonyms + dates, never model inputs |
| Feature (Mar 1–15 only) | `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_data_available` | all knowable before Mar 16 |
| Label / proxy | `gsc_impressions` summed over Mar 16–31 | this is what I'm trying to predict, never a feature |
| Excluded | `fact_content_query_90d` | window-overlap risk, not checked yet |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1** — grain (proves "one row = one page, one client, one day")

In [74]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS dup_cnt
    FROM {TABLES['fact_daily_march']}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").df()
print(f"Duplicate (date, client, content) rows found: {len(grain_check)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (date, client, content) rows found: 0


**Query 2** — row count and date span:

In [75]:
#span and volume check
span_df = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT client_hash_id) AS total_clients,
        COUNT(DISTINCT content_hash_id) AS total_urls
    FROM {TABLES['fact_daily_march']}
""").df()
display(span_df)

,total_rows,min_date,max_date,total_clients,total_urls
0,9841378,2026-03-01,2026-03-31,55,331437


**Query 3** — availability, filtered with IS TRUE:

In [76]:
#  Availability check with IS TRUE (Strict verification)
total_rows = int(span_df["total_rows"][0])
available = con.sql(f"""
    SELECT COUNT(*)
    FROM {TABLES['fact_daily_march']}
    WHERE ga4_data_available IS TRUE
""").fetchone()[0]

print(f"Total March rows:        {total_rows:,}")
print(f"Rows with GA4 available: {available:,} ({100*available/total_rows:.1f}%)")

availability_df = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, ga4_data_available
    FROM {TABLES['fact_daily_march']}
    WHERE ga4_data_available IS TRUE
    LIMIT 5
""").df()
display(availability_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total March rows:        9,841,378
Rows with GA4 available: 413,966 (4.2%)


,report_date,client_hash_id,content_hash_id,ga4_data_available
0,2026-03-01,client_65de48885f4ef01b,content_09be8cc7fcb222af,True
1,2026-03-01,client_65de48885f4ef01b,content_851afac9fe13612e,True
2,2026-03-01,client_65de48885f4ef01b,content_cee6c6fc8c51af14,True
3,2026-03-01,client_65de48885f4ef01b,content_5e120e972f11f833,True
4,2026-03-01,client_65de48885f4ef01b,content_16a7291bb6ecaebe,True


**Feature Frame & Leakage Trap Experiment**

In [77]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [78]:
#1. Feature Generation (Mar 1–15) - Aliased cleanly to match Python
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_first_half,
        AVG(gsc_avg_position) FILTER (WHERE gsc_impressions > 0) AS avg_position_first_half,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions_first_half,
        AVG(CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0.0 END) AS ga4_available_share_first_half,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS first_half_ctr
    FROM {TABLES['fact_daily_march']}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 10
""").df()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [79]:
print(f"{len(features):,} content items with >=10 impressions in Mar 1-15")
features.head()

120,513 content items with >=10 impressions in Mar 1-15


,client_hash_id,content_hash_id,impressions_first_half,avg_position_first_half,days_with_impressions_first_half,ga4_available_share_first_half,first_half_ctr
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,6.327311,15,0.0,0.001438
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,3.906852,15,0.0,0.000000
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,6.473735,15,0.0,0.000810
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,7.259861,15,0.0,0.003279
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,14.0,9.000000,9,0.0,0.000000


- **impressions_first_half** — knowable because it only sums days already in the past relative to the Mar 16 decision point.
- **avg_position_first_half** — same window; filtered to days with impressions so zero-impression days don't drag the average down.
- **days_with_impressions_first_half** — a consistency signal; only counts days already elapsed at decision time.
- **ga4_available_share_first_half** — whether GA4 tracking exists yet for this page, built only from the flag as it stood through Mar 15.
- **first_half_ctr** — clicks ÷ impressions, both restricted to Mar 1–15; a ratio of two already-knowable numbers is still knowable.

In [80]:
# 2. Outcome Window (Mar 16–31)
labels = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_h2
    FROM {TABLES['fact_daily_march']}
    WHERE report_date >= DATE '2026-03-16'
    GROUP BY 1, 2
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [81]:
#3. Merge & Construct Binary Target
df = features.merge(labels, on=["client_hash_id", "content_hash_id"], how="left")
df["imp_h2"] = df["imp_h2"].fillna(0)
df["is_declining"] = (df["imp_h2"] < 0.80 * df["impressions_first_half"]).astype(int)

feature_cols = [
    "impressions_first_half",
    "avg_position_first_half",
    "days_with_impressions_first_half",
    "ga4_available_share_first_half",
    "first_half_ctr"
]
df_clean = df.dropna(subset=feature_cols)

X = df_clean[feature_cols]
y = df_clean["is_declining"]

In [82]:
# 4. Honest Model
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_honest.fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, rf_honest.predict_proba(X_te)[:, 1])
print(f"HONEST ROC-AUC: {honest_auc:.4f}")

HONEST ROC-AUC: 0.6155


In [83]:
# 5. The Trap (Leakage Test)
X_leaky = df_clean[feature_cols + ["imp_h2"]]
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leaky, y, test_size=0.2, random_state=42, stratify=y)
rf_leaky = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_leaky.fit(X_tr_l, y_tr_l)
leaky_auc = roc_auc_score(y_te_l, rf_leaky.predict_proba(X_te_l)[:, 1])
print(f"LEAKY ROC-AUC:  {leaky_auc:.4f}  (Score Jump: {honest_auc:.4f} -> {leaky_auc:.4f})")

LEAKY ROC-AUC:  0.9991  (Score Jump: 0.6155 -> 0.9991)


In [84]:
# Cleanup leaky artifacts
del X_leaky, X_tr_l, X_te_l, y_tr_l, y_te_l, rf_leaky
print(f"Cleanup complete. Retained honest baseline AUC: {honest_auc:.4f}")

Cleanup complete. Retained honest baseline AUC: 0.6155


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [85]:
late_starts = con.sql(f"""
    SELECT COUNT(*) FROM {TABLES['dim_clients']}
    WHERE gsc_data_start > DATE '2026-03-01'
""").fetchone()[0]

total_clients = con.sql(f"SELECT COUNT(*) FROM {TABLES['dim_clients']}").fetchone()[0]
print(f"{late_starts} of {total_clients} clients started GSC tracking after 2026-03-01")

15 of 104 clients started GSC tracking after 2026-03-01


 **SLICE LIMITATIONS:**

1. **Onboarding Truncation:** Clients onboarded mid-month (GSC start date > 2026-03-01) produce artificially low H1 impression baselines, leading to false degradation flags if unadjusted.
2. **Confounding Factors:** An impression drop in H2 does not isolate actual content degradation from seasonal query shifts or page URL/canonical restructures.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.